# 11 — Blended Wind Forcing (ERA5 + in-situ stations)

ERA5 resolution is ~30 km, so only ~1 pixel covers the Stagnone lagoon → local wind effects (channeling by Isola Lunga, topographic modifications) are lost.

**Blending strategy:**
1. Build a finer output grid (~500 m) covering the ERA5 extent
2. Inside the lagoon: inverse-distance weighting (IDW) of the 2 in-situ stations (AE + Mulino)
3. Outside the transition zone: ERA5 bilinear-interpolated
4. Transition zone: smooth weighted blend
5. Output new NetCDF files replacing `era5_u10n_*` and `era5_v10n_*` in the v02 ext file

**Note:** only `u10n` and `v10n` are replaced. `msl` and `chnk` stay as ERA5 (no local measurements).

## 1. Imports and paths

In [ ]:
%matplotlib inline
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import xarray as xr
from scipy.interpolate import RegularGridInterpolator

project_root = Path(r'F:\StagnoneDT')
v02_dir = project_root / 'model' / 'dflowfm_v02'
processed_dir = project_root / 'data' / 'processed'
figures_dir = project_root / 'figures'

# Station locations (in-situ wind)
STATIONS = {
    'AE': {'lon': 12.447, 'lat': 37.890},
    'Mulino': {'lon': 12.482, 'lat': 37.868},
}

# Lagoon influence area
LAGOON_CENTER = (12.462, 37.867)
INNER_RADIUS_KM = 3.0   # fully IDW from stations
OUTER_RADIUS_KM = 8.0   # fully ERA5 beyond this
# Between inner and outer: linear blend

# Output grid resolution (degrees)
DX_OUT = 0.005  # ~500 m
DY_OUT = 0.005

## 2. Load ERA5 wind and in-situ wind

In [ ]:
# Load ERA5 from v02 dir
era5_u = xr.open_dataset(str(v02_dir / 'era5_u10n_20250701to20250710_ERA5.nc'))
era5_v = xr.open_dataset(str(v02_dir / 'era5_v10n_20250701to20250710_ERA5.nc'))

print(f'ERA5 grid: {era5_u.sizes["latitude"]} lat x {era5_u.sizes["longitude"]} lon')
print(f'  lat: {era5_u.latitude.values}')
print(f'  lon: {era5_u.longitude.values}')
print(f'  time: {era5_u.time.values[0]} to {era5_u.time.values[-1]}')
print(f'  u10n range: {float(era5_u.u10n.min()):.2f} to {float(era5_u.u10n.max()):.2f} m/s')
print(f'  v10n range: {float(era5_v.v10n.min()):.2f} to {float(era5_v.v10n.max()):.2f} m/s')

In [ ]:
# Load in-situ wind (10-min UTC)
ae = pd.read_csv(str(processed_dir / 'wind_AE_10min_UTC.csv'), index_col=0, parse_dates=True)
mul = pd.read_csv(str(processed_dir / 'wind_Mulino_10min_UTC.csv'), index_col=0, parse_dates=True)

print(f'AE: {len(ae)} records, columns: {list(ae.columns)}')
print(f'Mulino: {len(mul)} records')
print(f'\nAE range (speed at 10m): {ae.speed_10m.min():.1f} to {ae.speed_10m.max():.1f} m/s')
print(f'Mulino range (speed at 10m): {mul.speed_10m.min():.1f} to {mul.speed_10m.max():.1f} m/s')

In [ ]:
# Convert in-situ direction + speed to u,v components (eastward, northward)
# Convention: direction is FROM, 0=N, 90=E (meteorological)
# u = -speed * sin(dir_rad) = eastward wind vector component
# v = -speed * cos(dir_rad) = northward wind vector component
def dir_speed_to_uv(speed, direction_deg):
    rad = np.radians(direction_deg)
    u = -speed * np.sin(rad)
    v = -speed * np.cos(rad)
    return u, v

ae['u'], ae['v'] = dir_speed_to_uv(ae['speed_10m'], ae['dir_deg'])
mul['u'], mul['v'] = dir_speed_to_uv(mul['speed_10m'], mul['dir_deg'])

# Resample to hourly (matching ERA5 frequency) using vector mean
ae_h = ae[['u', 'v']].resample('1h').mean().dropna()
mul_h = mul[['u', 'v']].resample('1h').mean().dropna()

# Align on common times with ERA5
era5_times = pd.DatetimeIndex(era5_u.time.values)
ae_h = ae_h.reindex(era5_times, method='nearest', tolerance=pd.Timedelta('30min'))
mul_h = mul_h.reindex(era5_times, method='nearest', tolerance=pd.Timedelta('30min'))

print(f'AE hourly: {len(ae_h.dropna())} records after alignment')
print(f'Mulino hourly: {len(mul_h.dropna())} records after alignment')

## 3. Compare ERA5 vs in-situ at station locations

Verify the bias/pattern mismatch that motivates the blending.

In [ ]:
# Extract ERA5 at each station's nearest grid point
def era5_at_point(ds_u, ds_v, lon, lat):
    u = ds_u.u10n.sel(latitude=lat, longitude=lon, method='nearest').to_pandas()
    v = ds_v.v10n.sel(latitude=lat, longitude=lon, method='nearest').to_pandas()
    return u, v

era5_u_ae, era5_v_ae = era5_at_point(era5_u, era5_v, STATIONS['AE']['lon'], STATIONS['AE']['lat'])
era5_u_mul, era5_v_mul = era5_at_point(era5_u, era5_v, STATIONS['Mulino']['lon'], STATIONS['Mulino']['lat'])

era5_speed_ae = np.sqrt(era5_u_ae ** 2 + era5_v_ae ** 2)
era5_speed_mul = np.sqrt(era5_u_mul ** 2 + era5_v_mul ** 2)
insitu_speed_ae = np.sqrt(ae_h['u'] ** 2 + ae_h['v'] ** 2)
insitu_speed_mul = np.sqrt(mul_h['u'] ** 2 + mul_h['v'] ** 2)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(era5_speed_ae.index, era5_speed_ae, 'r-', linewidth=1, label='ERA5 (nearest pixel)')
axes[0].plot(insitu_speed_ae.index, insitu_speed_ae, 'b-', linewidth=1, label='In-situ AE (10m adj.)')
axes[0].set_title('AE — Wind speed: ERA5 vs in-situ')
axes[0].set_ylabel('Speed (m/s)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(era5_speed_mul.index, era5_speed_mul, 'r-', linewidth=1, label='ERA5 (nearest pixel)')
axes[1].plot(insitu_speed_mul.index, insitu_speed_mul, 'b-', linewidth=1, label='In-situ Mulino (10m adj.)')
axes[1].set_title('Mulino — Wind speed: ERA5 vs in-situ')
axes[1].set_ylabel('Speed (m/s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

fig.tight_layout()
plt.savefig(str(figures_dir / 'wind_era5_vs_insitu.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMean wind speed:')
print(f'  AE     : ERA5={era5_speed_ae.mean():.2f}, in-situ={insitu_speed_ae.mean():.2f} m/s')
print(f'  Mulino : ERA5={era5_speed_mul.mean():.2f}, in-situ={insitu_speed_mul.mean():.2f} m/s')

## 4. Build blended grid

Create a finer output grid covering the ERA5 extent. At each grid point:
- Compute distance to lagoon center
- Weight: 0 (full ERA5) if distance > OUTER_RADIUS, 1 (full in-situ IDW) if distance < INNER_RADIUS, linear blend in between
- Final value = (1-w) * ERA5 + w * IDW(in-situ stations)

In [ ]:
# Build output grid matching ERA5 extent but finer
lat_era5 = era5_u.latitude.values
lon_era5 = era5_u.longitude.values
lat_min, lat_max = lat_era5.min(), lat_era5.max()
lon_min, lon_max = lon_era5.min(), lon_era5.max()

lat_out = np.arange(lat_min, lat_max + DY_OUT/2, DY_OUT)
lon_out = np.arange(lon_min, lon_max + DX_OUT/2, DX_OUT)
print(f'Output grid: {len(lat_out)} lat x {len(lon_out)} lon = {len(lat_out)*len(lon_out)} cells')
print(f'  lat: {lat_min:.3f} to {lat_max:.3f} step {DY_OUT}')
print(f'  lon: {lon_min:.3f} to {lon_max:.3f} step {DX_OUT}')

In [ ]:
# Compute blending weight at each output grid point
def dist_km(lon1, lat1, lon2, lat2):
    # Simple flat-earth approx (OK for small area)
    dx = (lon1 - lon2) * 111.0 * np.cos(np.radians((lat1 + lat2) / 2))
    dy = (lat1 - lat2) * 111.0
    return np.sqrt(dx ** 2 + dy ** 2)

Lon2d, Lat2d = np.meshgrid(lon_out, lat_out)
d_to_lagoon = dist_km(Lon2d, Lat2d, LAGOON_CENTER[0], LAGOON_CENTER[1])

# Blend weight: 1 inside, 0 outside, linear between
w = np.clip((OUTER_RADIUS_KM - d_to_lagoon) / (OUTER_RADIUS_KM - INNER_RADIUS_KM), 0, 1)

# Visualize the weight field
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.pcolormesh(lon_out, lat_out, w, cmap='RdBu_r', shading='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='In-situ weight (0=ERA5, 1=in-situ)')
ax.scatter([STATIONS['AE']['lon']], [STATIONS['AE']['lat']], c='yellow', s=100, marker='*',
            edgecolor='black', label='AE station', zorder=5)
ax.scatter([STATIONS['Mulino']['lon']], [STATIONS['Mulino']['lat']], c='lime', s=100, marker='*',
            edgecolor='black', label='Mulino station', zorder=5)
ax.scatter([LAGOON_CENTER[0]], [LAGOON_CENTER[1]], c='red', s=80, marker='x', label='Lagoon center', zorder=5)
# Draw ERA5 grid points
ax.scatter(np.tile(lon_era5, len(lat_era5)), np.repeat(lat_era5, len(lon_era5)),
            c='black', s=30, marker='+', alpha=0.6, label='ERA5 pixels')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Blending weight field')
ax.legend(loc='upper right')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(str(figures_dir / 'wind_blend_weight.png'), dpi=150)
plt.show()

## 5. Compute blended u10n, v10n fields for every timestep

This is the core operation: for each hourly timestep, compute:
- ERA5 bilinear-interpolated to the fine grid
- IDW of the 2 in-situ stations over the fine grid
- Blend using the weight field

In [ ]:
def idw_at_grid(u_vals, v_vals, station_lons, station_lats, Lon2d, Lat2d, p=2):
    """IDW over the grid using given station values."""
    d = np.zeros((len(station_lons),) + Lon2d.shape)
    for i, (lon_s, lat_s) in enumerate(zip(station_lons, station_lats)):
        d[i] = dist_km(Lon2d, Lat2d, lon_s, lat_s) + 1e-3  # avoid div by 0
    w_idw = 1 / d ** p
    w_idw /= w_idw.sum(axis=0)
    u_grid = np.sum(w_idw * np.asarray(u_vals)[:, None, None], axis=0)
    v_grid = np.sum(w_idw * np.asarray(v_vals)[:, None, None], axis=0)
    return u_grid, v_grid

# Prepare arrays
nt = len(era5_u.time)
u_out = np.full((nt, len(lat_out), len(lon_out)), np.nan, dtype=np.float32)
v_out = np.full_like(u_out, np.nan)

station_lons = [STATIONS['AE']['lon'], STATIONS['Mulino']['lon']]
station_lats = [STATIONS['AE']['lat'], STATIONS['Mulino']['lat']]

print('Blending each timestep...')
for ti in range(nt):
    # ERA5 bilinear to fine grid
    u_e = era5_u.u10n.isel(time=ti).values
    v_e = era5_v.v10n.isel(time=ti).values
    # Ensure lat is ascending for interpolator (ERA5 typically descending)
    if lat_era5[0] > lat_era5[-1]:
        lat_src = lat_era5[::-1]
        u_e_src = u_e[::-1, :]
        v_e_src = v_e[::-1, :]
    else:
        lat_src = lat_era5
        u_e_src = u_e
        v_e_src = v_e
    interp_u = RegularGridInterpolator((lat_src, lon_era5), u_e_src, bounds_error=False, fill_value=None)
    interp_v = RegularGridInterpolator((lat_src, lon_era5), v_e_src, bounds_error=False, fill_value=None)
    pts = np.column_stack([Lat2d.ravel(), Lon2d.ravel()])
    u_e_fine = interp_u(pts).reshape(Lat2d.shape)
    v_e_fine = interp_v(pts).reshape(Lat2d.shape)
    
    # In-situ IDW: use available station data at this timestep
    t = era5_u.time.values[ti]
    u_ae = ae_h['u'].get(t, np.nan) if t in ae_h.index else np.nan
    v_ae = ae_h['v'].get(t, np.nan) if t in ae_h.index else np.nan
    u_mul = mul_h['u'].get(t, np.nan) if t in mul_h.index else np.nan
    v_mul = mul_h['v'].get(t, np.nan) if t in mul_h.index else np.nan
    
    u_insitu_fine = np.full_like(u_e_fine, np.nan)
    v_insitu_fine = np.full_like(v_e_fine, np.nan)
    if not (np.isnan(u_ae) or np.isnan(u_mul)):
        u_insitu_fine, v_insitu_fine = idw_at_grid(
            [u_ae, u_mul], [v_ae, v_mul], station_lons, station_lats, Lon2d, Lat2d)
    
    # Blend: where in-situ is NaN, fall back to ERA5
    if np.isnan(u_insitu_fine).all():
        u_out[ti] = u_e_fine
        v_out[ti] = v_e_fine
    else:
        u_out[ti] = (1 - w) * u_e_fine + w * u_insitu_fine
        v_out[ti] = (1 - w) * v_e_fine + w * v_insitu_fine

    if ti % 50 == 0:
        print(f'  {ti+1}/{nt} done')
print('Blending complete.')

## 6. Visualize a blended snapshot

In [ ]:
# Pick a representative timestep (middle of simulation)
ti_snap = nt // 2
speed_out = np.sqrt(u_out[ti_snap] ** 2 + v_out[ti_snap] ** 2)

# Get ERA5 bilinear at same timestep for comparison
u_e = era5_u.u10n.isel(time=ti_snap).values
v_e = era5_v.v10n.isel(time=ti_snap).values
if lat_era5[0] > lat_era5[-1]:
    u_e_src = u_e[::-1, :]; v_e_src = v_e[::-1, :]
else:
    u_e_src = u_e; v_e_src = v_e
interp_u = RegularGridInterpolator((lat_src, lon_era5), u_e_src, bounds_error=False, fill_value=None)
interp_v = RegularGridInterpolator((lat_src, lon_era5), v_e_src, bounds_error=False, fill_value=None)
u_era_fine = interp_u(np.column_stack([Lat2d.ravel(), Lon2d.ravel()])).reshape(Lat2d.shape)
v_era_fine = interp_v(np.column_stack([Lat2d.ravel(), Lon2d.ravel()])).reshape(Lat2d.shape)
speed_era = np.sqrt(u_era_fine ** 2 + v_era_fine ** 2)

vmin = min(speed_era.min(), speed_out.min())
vmax = max(speed_era.max(), speed_out.max())

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (title, sp, uf, vf) in zip(axes,
    [('ERA5 only', speed_era, u_era_fine, v_era_fine),
     ('Blended (ERA5+in-situ)', speed_out, u_out[ti_snap], v_out[ti_snap])]):
    im = ax.pcolormesh(lon_out, lat_out, sp, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
    step = 4
    ax.quiver(Lon2d[::step, ::step], Lat2d[::step, ::step],
              uf[::step, ::step], vf[::step, ::step], scale=200, width=0.003)
    ax.scatter([STATIONS['AE']['lon']], [STATIONS['AE']['lat']], c='red', s=60, marker='*', zorder=5)
    ax.scatter([STATIONS['Mulino']['lon']], [STATIONS['Mulino']['lat']], c='red', s=60, marker='*', zorder=5)
    plt.colorbar(im, ax=ax, label='Speed (m/s)', shrink=0.8)
    ax.set_title(f'{title}\n{pd.Timestamp(era5_u.time.values[ti_snap])}')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_aspect('equal')

fig.tight_layout()
plt.savefig(str(figures_dir / 'wind_blend_snapshot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Write blended NetCDFs

Output two new files in v02 dir following the ERA5 format so D-Flow FM reads them without changes to the parser.

In [ ]:
# Build new datasets mirroring ERA5 structure
time_coord = era5_u.time

ds_u_out = xr.Dataset(
    data_vars={
        'u10n': (('time', 'latitude', 'longitude'), u_out,
                  {'units': 'm s**-1', 'long_name': 'blended eastward wind at 10m'}),
    },
    coords={
        'time': time_coord,
        'latitude': ('latitude', lat_out, {'units': 'degrees_north'}),
        'longitude': ('longitude', lon_out, {'units': 'degrees_east'}),
    },
    attrs={'description': 'Blended ERA5 + in-situ wind (AE + Mulino), v02 build'},
)

ds_v_out = xr.Dataset(
    data_vars={
        'v10n': (('time', 'latitude', 'longitude'), v_out,
                  {'units': 'm s**-1', 'long_name': 'blended northward wind at 10m'}),
    },
    coords={
        'time': time_coord,
        'latitude': ('latitude', lat_out, {'units': 'degrees_north'}),
        'longitude': ('longitude', lon_out, {'units': 'degrees_east'}),
    },
    attrs={'description': 'Blended ERA5 + in-situ wind (AE + Mulino), v02 build'},
)

# Write
out_u_path = v02_dir / 'wind_blended_u10n_20250701to20250710.nc'
out_v_path = v02_dir / 'wind_blended_v10n_20250701to20250710.nc'
ds_u_out.to_netcdf(str(out_u_path))
ds_v_out.to_netcdf(str(out_v_path))

print(f'Wrote: {out_u_path} ({out_u_path.stat().st_size/1e6:.2f} MB)')
print(f'Wrote: {out_v_path} ({out_v_path.stat().st_size/1e6:.2f} MB)')

## 8. Update v02 ext file to use blended wind

In [ ]:
ext_new = v02_dir / 'Stagnone_dxy01_15m_new.ext'

with open(ext_new, 'r') as f:
    ext_text = f.read()

print('--- Before replacement ---')
# Show wind-related lines
for line in ext_text.splitlines():
    if 'wind' in line.lower() or 'u10n' in line.lower() or 'v10n' in line.lower() or 'forcingFile' in line.lower():
        print(f'  {line}')

# Replace the forcingFile names
ext_text_new = ext_text.replace(
    'era5_u10n_20250701to20250710_ERA5.nc',
    'wind_blended_u10n_20250701to20250710.nc'
).replace(
    'era5_v10n_20250701to20250710_ERA5.nc',
    'wind_blended_v10n_20250701to20250710.nc'
)

with open(ext_new, 'w') as f:
    f.write(ext_text_new)

print('\n--- After replacement ---')
for line in ext_text_new.splitlines():
    if 'wind' in line.lower() or 'u10n' in line.lower() or 'v10n' in line.lower():
        print(f'  {line}')

print(f'\nUpdated: {ext_new}')

## 9. Summary

**Changes made to v02:**
- New blended wind files: `wind_blended_u10n_*.nc`, `wind_blended_v10n_*.nc` (~500 m resolution)
- Inside lagoon (< 3 km from center): pure IDW of AE + Mulino stations
- Outside transition zone (> 8 km): pure ERA5
- Smooth blend in between
- Ext file updated to reference the new files
- msl and chnk still from ERA5 (no local measurements)

**To run v02:**
```
cd model/dflowfm_v02
run_model.bat
```

Expected runtime: similar to v01 (~2h 45min).